# Temporal Crop Stability Experiments

Ce notebook reprend le script `temporal_crop_stability_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Stabilite des modeles crop temporels candidats streaming.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Repeated split validation for temporal crop attention/PPE models.
- Commande de reproduction referencee : temporal crop stability.
- Artefacts controles : Short temporal crop/clip repeated split stability exists. (`runs/exp_032_temporal_crop_stability/metrics/temporal_crop_stability_summary.csv`).
- Run par defaut : `runs/exp_032_temporal_crop_stability`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "temporal_crop_stability_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from crop_cnn_experiments import TARGETS
from crop_cnn_stability_experiments import parent_combo_split
from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, make_run_dir
from temporal_crop_experiments import (
    TemporalCropDataset,
    evaluate_predictions,
    predict,
    train_one,
)


## Fonction `make_temporal_index`

Cette cellule definit `make_temporal_index`. Elle prepare une partie du script.

In [ ]:
def make_temporal_index(crop_index, seq_len):
    rows = []
    for video_id, group in crop_index.groupby("video_id", sort=False):
        group = group.sort_values("time_s").reset_index(drop=True)
        if group.empty:
            continue
        for end_pos in range(len(group)):
            start_pos = max(0, end_pos - seq_len + 1)
            seq = group.iloc[start_pos : end_pos + 1]
            if len(seq) < seq_len:
                pad = pd.concat([seq.iloc[[0]]] * (seq_len - len(seq)), ignore_index=True)
                seq = pd.concat([pad, seq], ignore_index=True)
            end = group.iloc[end_pos]
            rows.append(
                {
                    "video_id": video_id,
                    "split": end["split"],
                    "end_frame": int(end["frame"]),
                    "end_time_s": float(end["time_s"]),
                    "paths": "|".join(str(p) for p in seq["path"].tolist()),
                    "frames": "|".join(str(int(f)) for f in seq["frame"].tolist()),
                    "times_s": "|".join(f"{float(t):.3f}" for t in seq["time_s"].tolist()),
                    "attention": end["attention"],
                    "attention_label": int(end["attention_label"]),
                    "blouse": end["blouse"],
                    "blouse_label": int(end["blouse_label"]),
                }
            )
    return pd.DataFrame(rows)


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(metrics):
    rows = []
    group_cols = ["target", "architecture", "level", "split"]
    for keys, group in metrics.groupby(group_cols):
        target, architecture, level, split = keys
        rows.append(
            {
                "target": target,
                "architecture": architecture,
                "level": level,
                "split": split,
                "n_repeats": int(group["repeat_seed"].nunique()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "roc_auc_mean": float(group["roc_auc"].mean()),
                "f1_mean": float(group["f1"].mean()),
                "f1_std": float(group["f1"].std(ddof=0)),
                "balanced_accuracy_mean": float(group["balanced_accuracy"].mean()),
                "balanced_accuracy_std": float(group["balanced_accuracy"].std(ddof=0)),
            }
        )
    return pd.DataFrame(rows)


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    source_run = Path(args.crop_run)
    if not source_run.is_absolute():
        source_run = ROOT / source_run
    source_index = pd.read_csv(source_run / "features" / "crop_cnn_index.csv")
    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "crop_run": str(source_run),
            "seeds": args.seeds,
            "seq_len": args.seq_len,
            "architectures": args.architectures,
            "epochs": args.epochs,
            "patience": args.patience,
            "image_size": args.image_size,
            "temporal_aug": not args.no_temporal_aug,
            "split_policy": "parent video split stratified by attention/blouse combination; crop sequences inherit parent split",
        },
    )
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_metrics = []
    all_history = []
    split_rows = []
    for seed in args.seeds:
        split = parent_combo_split(source_index, seed)
        split_crop = source_index.copy()
        split_crop["split"] = split_crop["video_id"].map(split)
        seq_index = make_temporal_index(split_crop, args.seq_len)
        seq_index.to_csv(run_dir / "features" / f"temporal_crop_split_seed_{seed}.csv", index=False)
        video_split = seq_index.groupby(["split", "video_id"]).size().reset_index()
        split_rows.append({"seed": seed, **video_split["split"].value_counts().to_dict()})
        for target in TARGETS:
            for architecture in args.architectures:
                print(f"training seed{seed} temporal crop {target} {architecture}")
                model, history, train_time_s, model_size = train_one(run_dir, source_run, seq_index, target, architecture, args, device)
                model_path = run_dir / "models" / f"{target}_{architecture}.pt"
                renamed = run_dir / "models" / f"{target}_seed{seed}_{architecture}.pt"
                if model_path.exists():
                    model_path.replace(renamed)
                for row in history:
                    row["repeat_seed"] = seed
                all_history.extend(history)
                eval_ds = TemporalCropDataset(source_run, seq_index, target, train=False, image_size=args.image_size, temporal_aug=False)
                eval_loader = DataLoader(eval_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)
                probs, _ = predict(model, eval_loader, device)
                pred = seq_index[["video_id", "split", "end_frame", "end_time_s", f"{target}_label"]].copy()
                pred["target"] = target
                pred["architecture"] = architecture
                pred["repeat_seed"] = seed
                pred["risk"] = probs
                pred.to_csv(run_dir / "features" / f"predictions_{target}_seed{seed}_{architecture}.csv", index=False)
                for metric in evaluate_predictions(pred, target):
                    metric.update(
                        {
                            "target": target,
                            "architecture": architecture,
                            "repeat_seed": seed,
                            "train_time_s": float(train_time_s),
                            "model_size_bytes": int(model_size),
                        }
                    )
                    all_metrics.append(metric)
                pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "temporal_crop_stability_metrics.csv", index=False)
                pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "temporal_crop_stability_training_history.csv", index=False)

    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "temporal_crop_stability_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "temporal_crop_stability_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "temporal_crop_stability_split_counts.csv", index=False)
    summary = summarize(metrics)
    summary.to_csv(run_dir / "metrics" / "temporal_crop_stability_summary.csv", index=False)

    lines = ["# Temporal Crop Repeated Split Stability", ""]
    lines.append("Parent videos are re-split by attention/blouse label combination. Temporal crop sequences inherit the parent split.")
    lines.append("")
    lines.append("| target | architecture | level | split | AP mean | AP std | F1 mean | balanced acc mean |")
    lines.append("|---|---|---|---|---:|---:|---:|---:|")
    for _, row in summary.sort_values(["target", "level", "split", "ap_mean"], ascending=[True, True, True, False]).iterrows():
        lines.append(
            f"| {row['target']} | {row['architecture']} | {row['level']} | {row['split']} | "
            f"{row['ap_mean']:.3f} | {row['ap_std']:.3f} | {row['f1_mean']:.3f} | {row['balanced_accuracy_mean']:.3f} |"
        )
    summary_path = run_dir / "temporal_crop_stability_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(
        run_dir,
        "Temporal Crop Repeated Split Stability",
        f"- Seeds: `{args.seeds}`\n- Summary: `{summary_path}`",
    )
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Repeated split validation for temporal crop attention/PPE models.")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--run-name", default="exp_032_temporal_crop_stability")
    parser.add_argument("--architectures", nargs="+", default=["mean_pool", "flat_mlp", "gru", "tcn"])
    parser.add_argument("--seeds", nargs="+", type=int, default=[111, 222, 333])
    parser.add_argument("--seq-len", type=int, default=4)
    parser.add_argument("--image-size", type=int, default=112)
    parser.add_argument("--epochs", type=int, default=8)
    parser.add_argument("--patience", type=int, default=3)
    parser.add_argument("--batch-size", type=int, default=24)
    parser.add_argument("--lr", type=float, default=4e-4)
    parser.add_argument("--weight-decay", type=float, default=2e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-temporal-aug", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_032_temporal_crop_stability_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["temporal_crop_stability_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
